# DysXAI — Initialization (00)

## Recommended run order (notebooks)

| Step | Notebook | Purpose |
|------|----------|---------|
| 00 | `00_initialization.ipynb` | Data paths, load `meta_df`, shared helpers (`dysxai_init.py`) |
| 01 | `01_model_cnn1d.ipynb` | 1D CNN |
| 02 | `02_model_tcn.ipynb` | TCN |
| 03 | `03_model_cnnlstm.ipynb` | CNN–LSTM |
| 04 | `04_feature_selection.ipynb` | Forward feature selection |
| 05 | `05_explainability.ipynb` | XAI / ablations |
| 06 | `06_padding_ablation.ipynb` | Time / padding leakage ablations |
| 07 | `07_cv_evaluation.ipynb` | Subject-stratified K-fold evaluation |
| 08 | `08_analyze_true_duration.ipynb` | True duration stats vs labels / age |

**Scripts:** The `.py` files in the project root are the **canonical** implementations; this notebook imports them and shows full source below for reading.

**Legacy:** `modelDysgraphia6a.ipynb` is an older monolithic draft — use the numbered pipeline instead.


In [ ]:
# Optional: install dependencies (safe to re-run)
import subprocess
import sys


def need_install(name: str) -> bool:
    try:
        __import__(name)
        return False
    except ImportError:
        return True


to_install = []
for mod, pip in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("torch", "torch"),
    ("sklearn", "scikit-learn"),
    ("matplotlib", "matplotlib"),
    ("tqdm", "tqdm"),
    ("seaborn", "seaborn"),
    ("scipy", "scipy"),
]:
    if need_install(mod):
        to_install.append(pip)
try:
    import openpyxl  # noqa: F401
except ImportError:
    to_install.append("openpyxl")

if to_install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *to_install, "--quiet"])
    print("Installed:", to_install)
else:
    print("Core dependencies OK.")


Core dependencies OK.


## Load shared module and build meta_df

Implementation lives in **dysxai_init.py**. Edit paths via optional overrides in the next cell.

**Pipeline (data -> model input):** raw **X** / **Y** are smoothed with a 2nd-order Butterworth low-pass (defaults: 12 Hz cutoff, Config.SAMPLING_RATE_HZ = 133) before velocity, acceleration, and jerk; **Time** is used for dt then dropped if Config.DROP_TIME_CHANNEL. Subject **Age** (from metadata) is broadcast to every timestep as the last channel. Use get_model_input_channel_count() and model_channel_names() so notebooks stay in sync with dysxai_init.py.



In [2]:
import os
import sys
from pathlib import Path


def project_root() -> Path:
    """Directory containing dysxai_init.py (search cwd and parents)."""
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "dysxai_init.py").is_file():
            return p
    return here


ROOT = project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import dysxai_init
from dysxai_init import *  # noqa: F403 — Config, HandwritingDataset, helpers

# Optional: override paths if data are not under the project folder
# dysxai_init.Config.DATA_ROOT = r"C:\\path\\to\\dataSciRep_public"
# dysxai_init.Config.META_XLSX = r"C:\\path\\to\\data2_SciRep_pub.xlsx"

meta_df = dysxai_init.run_init(verbose=True)
print("Device:", dysxai_init.Config.DEVICE)
print("Samples:", len(meta_df), "| Subjects:", meta_df["subject_id"].nunique())
print("Model input channel count:", dysxai_init.get_model_input_channel_count())
print("Channel order (post-load):", dysxai_init.model_channel_names())


Looking for files in: C:\Users\tiama\OneDrive\Desktop\Reserach Project\DysXAI\dysxai_tasks_split\task_5_hrackarstvo
Loaded 120 samples from 120 subjects
Device: cpu
Samples: 120 | Subjects: 120


## Data directory check


In [3]:
# Quick sanity check on data directory
import os
import dysxai_init

dr = dysxai_init.Config.DATA_ROOT
print("DATA_ROOT exists:", os.path.isdir(dr), "|", dr)


DATA_ROOT exists: True | C:\Users\tiama\OneDrive\Desktop\Reserach Project\DysXAI\dysxai_tasks_split\task_5_hrackarstvo


## `dysxai_init.py` — full file (reference)


In [4]:
# Full source of dysxai_init.py (reference). Executed logic is imported above.
from pathlib import Path

try:
    from IPython.display import Code, display

    display(Code(filename=str(ROOT / "dysxai_init.py"), language="python"))
except ImportError:
    print((ROOT / "dysxai_init.py").read_text(encoding="utf-8"))


"""
Shared initialization module for DysXAI project.
Import this when running model notebooks without having run 00_initialization.ipynb first.
"""

import os
import re
import numpy as np
import pandas as pd
from typing import Any, Dict, List, Optional, Tuple, Union

import torch
from torch.utils.data import Dataset
from sklearn.preprocessing import StandardScaler


# --- Config ---
class Config:
    """Global configuration for the dysgraphia detection project."""
    _PROJECT_ROOT = os.path.dirname(os.path.abspath(__file__))
    # Full-session recordings (legacy): nested public release tree.
    RAW_DATASCI_ROOT = os.path.join(_PROJECT_ROOT, "dataSciRep_public", "dataSciRep_public")
    # Task 5 = demanding word "hračkárstvo" (Drotár); single-task clips reduce cross-task mixing.
    TASK_SPLIT_SUBDIR = os.path.join("dysxai_tasks_split", "task_5_hrackarstvo")
    DATA_ROOT = os.path.join(_PROJECT_ROOT, TASK_SPLIT_SUBDIR)
    META_XLSX = os.path.join(_PROJECT_ROOT, "data2_SciRep_pub.xlsx")
    MAX_LEN = 2000
    USE_DERIVATIVES = True
    # Remove Time (raw index 2) after derivatives so the CNN cannot exploit padding/length via the clock.
    # Derivatives are still computed using timestamps internally, then Time is dropped before scaling/padding.
    DROP_TIME_CHANNEL = True
    # Age-adjusted Z-scores: per-age mean/std from training controls only (reduces age vs. pathology confound).
    USE_AGE_ADJUSTED_SCALING = True
    # Skip thin buckets; those ages fall back to global train-control stats.
    AGE_Z_MIN_TIMESTEPS_PER_BUCKET = 50
    AGE_Z_MIN_CONTROL_FILES_PER_BUCKET = 2
    BATCH_SIZE = 16
    NUM_EPOCHS = 50
    LR = 1e-3
    WEIGHT_DECAY = 1e-4
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    TRAIN_SUBJECT_RATIO = 0.7
    VAL_SUBJECT_RATIO = 0.15
    TEST_SUBJECT_RATIO = 0.15
    RANDOM_STATE = 42
    EARLY_STOPPING_PATIENCE = 5
    EARLY_STOPPING_ENABLED = True


# Channel order after compute_derivatives (USE_DERIVATIVES=True): matches 04_feature_selection FEATURE_NAMES
CHANNEL_NAMES = [
    "X", "Y", "Time", "Pressure", "Azimuth", "Altitude", "Pen_Status",
    "Velocity_X", "Velocity_Y", "Acceleration_X", "Acceleration_Y", "Jerk_X", "Jerk_Y",
]
TIME_CHANNEL_INDEX = 2


def get_model_input_channel_count() -> int:
    """Channels after optional derivatives and optional Time drop (what Conv1d sees)."""
    n = 7 + (6 if Config.USE_DERIVATIVES else 0)
    if Config.DROP_TIME_CHANNEL:
        n -= 1
    return n


def model_channel_names() -> List[str]:
    """Human-readable names per model input channel (same order as tensors after load_and_process_timeseries)."""
    names = list(CHANNEL_NAMES) if Config.USE_DERIVATIVES else list(CHANNEL_NAMES[:7])
    if Config.DROP_TIME_CHANNEL and len(names) > TIME_CHANNEL_INDEX:
        names.pop(TIME_CHANNEL_INDEX)
    return names


def drop_time_channel(ts: np.ndarray, time_index: int = TIME_CHANNEL_INDEX) -> np.ndarray:
    """Remove the Time column from a (T, C) array (post-derivatives layout when USE_DERIVATIVES)."""
    if ts.ndim != 2 or ts.shape[1] <= time_index:
        return ts
    return np.delete(ts, time_index, axis=1).astype(np.float32)


def load_and_process_timeseries(filepath: str, use_derivatives: bool | None = None) -> np.ndarray:
    """
    Load raw .svc, optionally append velocity/acceleration/jerk (using Time for dt), optionally drop Time.

    Time is dropped *after* derivatives so kinematics stay physically scaled; dropped *before* scaler/padding.
    """
    ud = Config.USE_DERIVATIVES if use_derivatives is None else use_derivatives
    ts = load_raw_timeseries(filepath)
    if ud:
        ts = compute_derivatives(ts)
    if Config.DROP_TIME_CHANNEL:
        ts = drop_time_channel(ts)
    return ts.astype(np.float32)


def load_metadata(meta_path: str) -> pd.DataFrame:
    """Load metadata from Excel file (includes age for age-adjusted scaling)."""
    df = pd.read_excel(meta_path)
    df["subject_id"] = df["ID"].astype(int)
    df["label"